[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/04_phase2_stage3_multiasset_reference.ipynb)

# 04 · Stage 3: multi-asset ground truth

**Phase:** Phase 2: multi-asset

**Goal:** Exact 2-asset common-jump density vs a 2-D finite-difference solver vs Monte Carlo; how grid cost explodes with dimension (RQ4 motivation) and how Monte Carlo behaves as d grows.

**Outputs:** `figures/stage3_multiasset_reference/fig01–fig04`, `results/stage3_multiasset_reference/metrics.json`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive, install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # keep figures/results across Colab sessions (recommended)
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    rc = os.system(f'git -C "{ROOT}" pull -q --ff-only')
    if rc != 0:   # usually: result files on Drive that have since been committed to GitHub
        print("WARNING: git pull failed; the Drive copy is out of date. To sync (keeps untracked files), run:")
        print(f'!git -C "{ROOT}" fetch -q origin && git -C "{ROOT}" reset -q --hard origin/main')
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## 1. Configuration (`configs/stage3_multiasset.yaml`)
Jump parameters, t0 and T come from `configs/base.yaml`.

In [ ]:
print(open('configs/stage3_multiasset.yaml').read())

## 2. Run the stage
CPU only (no GPU needed), about 5–10 min: the 2-D grid convergence study dominates.

In [ ]:
!python scripts/stage3_multiasset_reference.py

## 3. Figures and what to check

| Figure | Pass criterion |
|---|---|
| fig01_density_2d | FD contours (orange) sit on the exact contours (dashed); the common-jump ridge is visible; FD log-error small |
| fig02_fd_cost | (a) L¹ falls with slope ≈ 2; (b) wall time grows with the number of unknowns; (c) one grid exceeds GPU memory beyond d = 3 |
| fig03_mc_vs_d | MC error at fixed N roughly flat in d; time per query grows with d. **The PINN has to win on latency, not accuracy** |
| fig04_portfolio_es | FD and MC agree with the exact 1-D projection for every weight; the ES curve shows the diversification benefit |

In [ ]:
show_figures("stage3_multiasset_reference")

## 4. Numbers

In [ ]:
m = load_metrics("stage3_multiasset_reference")
print("reference 2-D FD:", m["fd2d_reference"])
display(pd.DataFrame(m["fd2d_convergence"]).style.format(precision=3))
display(pd.DataFrame(m["mc_vs_d"]).style.format(precision=4))
w = pd.DataFrame(m["portfolio_weights"])
w["fd_rel"] = (w.es_fd - w.es_exact) / w.es_exact; w["mc_rel"] = (w.es_mc - w.es_exact) / w.es_exact
display(w.style.format(precision=5))